## 初期設定

In [1]:
# 更新履歴
## 2026/01/23 新peakfitに適合するようにしました
## 2026/01/23 imshowをlogで表示するようにしました
## 2026/01/22 ピークフィットの結果をcsvとして保存できるようにしました
## 2026/01/05 save_tiffの追加
## 2025/12/29 disp_cakingの作成
## 2025/12/17 plotlyを組み込み
## 2025/12/17 作成

# 基本モジュール
import sys, os
from IPython.display import display, HTML, clear_output, update_display, Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import pyFAI
import pandas as pd
import h5py
import plotly.express as px
import plotly.graph_objects as go
from tqdm import tqdm
import PIL.Image as im

# 自作モジュール
sys.path.append(r"C:\Users\okaza\pythonenv")
from modules.Mytools.handle_ipynb import save_pickle, load_pickle, export_html, ask_openfilename, ask_savefilename
from modules.Mytools.Tools import h5_tree, dict_tree, his2array
from modules.Mytools.PseudoVoigt import peakfit, pseudoVoigt
import modules.Mytools.Settings
sys.path.append(os.getcwd())
from main import Create_Hdfdata # type: ignore

# 初期パラメーター
cachedir = os.path.join(os.getcwd(), ".cache")
os.makedirs(cachedir, exist_ok=True)

## 初期化

In [2]:
Experiment = "UODE36_0023"

In [3]:
# Create_Hdfdataのインスタント化
i2a = Create_Hdfdata(
    name = Experiment,
    cachedir = cachedir,
    log = True
)

name: UODE36_0023
cachedir: d:\DATA\SPring-8-2025-Dec\Analysis\FPD\create_hdfdata\.cache
log: True
version: 1.0


## ファイルリストの作成

In [4]:
# ディレクトリ名を指定
dir = r"D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_2"

# ヘッダーとフッターを指定
header = "UODE36_23_"
footer = ".his"

In [5]:
# ファイルリストの格納
def get_filelist(dir: str,
                 header: str,
                 footer: str
                 ) -> list:
    
    print("="*70)
    print("Inputs:")
    print("\tdir:    " + dir)
    print("\theader: " + header)
    print("\tfooter: " + footer)
    print("="*70)

    ## ヘッダーとフッターを含むファイル名を取得
    flist = list()
    for __ in os.listdir(dir):
        if not header in __:
            continue
        if not footer in __:
            continue
        flist.append(__)

    ## ソート
    flist.sort(key = (lambda x: int(x.replace(header, "").replace(footer, ""))))
    filenames = list(map(lambda x: os.path.join(dir, x), flist))

    ## 表示
    for f in filenames:
        print(f)
    
    return filenames

## 格納
i2a.filelist = get_filelist(dir, header, footer)
del get_filelist

Inputs:
	dir:    D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_2
	header: UODE36_23_
	footer: .his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_2\UODE36_23_0.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_2\UODE36_23_1.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_2\UODE36_23_2.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_2\UODE36_23_3.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_2\UODE36_23_4.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_2\UODE36_23_5.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_2\UODE36_23_6.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_2\UODE36_23_7.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_2\UODE36_23_8.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_2\UODE36_23_9.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_2\UODE36_23_10.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_2\UODE36_23_11.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_2\UODE36_23_12.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_2\UODE36_23_13.his
D:\DATA\SPring-8

## 一次元化

In [7]:
import warnings

In [8]:
warnings.filterwarnings("ignore")

In [6]:
# 校正用ファイル
poni = r"D:\DATA\SPring-8-2025-Dec\ceo2_20251128_EH2_NB\IPAnalyzer_IP_MgS400_EH2_20251215_YusukeOkazaki.poni"

# step
step = 0.005

# 2thetaの範囲
radial_range = (3,30)

In [9]:
# 一気に1次元化する
hdf_1d = i2a.integrate1D(
    poni = poni,
    npt_rad = int((radial_range[1]-radial_range[0])/step),
    radial_range = radial_range,
)

  0%|          | 0/154 [00:00<?, ?it/s]

100%|██████████| 154/154 [00:00<00:00, 10374.10it/s]


Progress: [■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■] 100% (154/154) 

Integration completed.
<HDF5 file "UODE36_0023_integrate1d.hdf" (mode r)>
├── integrated
│   ├── frame = 0 ((5400,), float32)
│   ├── frame = 1 ((5400,), float32)
│   ├── frame = 10 ((5400,), float32)
│   ├── frame = 100 ((5400,), float32)
│   ├── frame = 101 ((5400,), float32)
│   ├── frame = 102 ((5400,), float32)
│   ├── frame = 103 ((5400,), float32)
│   ├── frame = 104 ((5400,), float32)
│   ├── frame = 105 ((5400,), float32)
│   ├── frame = 106 ((5400,), float32)
│   ├── frame = 107 ((5400,), float32)
│   ├── frame = 108 ((5400,), float32)
│   ├── frame = 109 ((5400,), float32)
│   ├── frame = 11 ((5400,), float32)
│   ├── frame = 110 ((5400,), float32)
│   ├── frame = 111 ((5400,), float32)
│   ├── frame = 112 ((5400,), float32)
│   ├── frame = 113 ((5400,), float32)
│   ├── frame = 114 ((5400,), float32)
│   ├── frame = 115 ((5400,), float32)
│   ├── frame = 116 ((5400,), float32)
│   ├── frame = 117 ((5400,)

結果を確認する

In [8]:
# hdfファイルの指定
# hdf_disp_caking = hdf_1d
hdf_disp_caking = r"D:\DATA\SPring-8-2025-Dec\Analysis\FPD\create_hdfdata\.cache\UODE36_0024_integrate1d.hdf"

# 表示する角度幅
rad_range = (13.7,13.75)
rad_range = (12,14)
rad_range = (11,14)

# フィッティングするか否か
flag_fit = True
flag_fit = False

# フィッティング範囲
fit_range = rad_range
# fit_range = (13.55, 14) # 011
fit_range = (11.9, 12.3) # 010

# plotly or matplotlibを表示する
flag_plotly = True # plotly
# flag_plotly = False # matplotlib

In [9]:
# 一次元化の結果を表示する
def disp_caking(hdf: str,
                rad_range: tuple[float,float],
                flag_fit: bool,
                fit_range: tuple[float, float],
                flag_plotly: bool,
                ) -> None:
    
    # 引数出力
    print("="*70)
    print("Inputs:")
    print("\thdf:         " + hdf)
    print("\trad_range:   {}-{}".format(*rad_range))
    print("\tflag_fit:    {}".format(flag_fit))
    print("\tfit_range:   {}-{}".format(*fit_range))
    print("\tflag_plotly: {}".format(flag_plotly))
    print("="*70)

    # データ読み込み
    d = []
    with h5py.File(hdf, mode = "r") as f:
        tth = np.array(f["rad"][()]) # type: ignore
        n_frame = len(f["integrated"].keys()) # type: ignore
        for i in range(n_frame):
            d.append(np.array(f["integrated/frame = {}".format(i)][()])) # type: ignore
    d = np.stack(d)

    # peakfit
    fit_mask = (tth > fit_range[0])&(tth < fit_range[1])
    if flag_fit:
        pf = peakfit()
        peaks = []
        popts = []
        errs = []
        for i in tqdm(range(n_frame)):
            tth_fit = tth[fit_mask]
            insty_fit = d[i][fit_mask]
            fit_res = pf.fit(x = tth_fit, y = insty_fit)
            peaks.append(fit_res[0][3])
            popts.append(fit_res[0])
            errs.append(fit_res[2][0]["Error"])
        peaks = np.array(peaks)
        xs = np.linspace(*fit_range, 200)
        df = pd.DataFrame(popts)
        df.columns = pf.variables()
        df["err"] = errs
        csvfilename = os.path.join(cachedir, i2a.name + "_popts.csv")
        df.to_csv(csvfilename)
        print("[Save csv]: " + csvfilename)

    # imshow用データの作成
    disp_mask = (tth > rad_range[0])&(tth < rad_range[1])
    disp_tth = tth[disp_mask]
    disp_d = d.T[disp_mask].T

    if True:
        # matplotlib
        if flag_fit:
            fig, axs = plt.subplots(3,1)
            fig.set_size_inches(6,12)
        else:
            fig, axs = plt.subplots(2,1)
            fig.set_size_inches(6,9)

        # plot
        aset = 5
        for i in range(n_frame):
            axs[0].plot(disp_tth, disp_d[i]+aset*i, lw = 0.1)
        axs[0].set_xlim(rad_range)
        axs[0].set_ylabel("Intensity [arb. unit]")

        # imshow
        axs[1].imshow(
            disp_d,
            origin = "lower",
            extent = (disp_tth[0], disp_tth[-1], 0, n_frame),
            aspect = "auto",
            norm = matplotlib.colors.LogNorm() # type: ignore
        )
        axs[1].set_xlim(rad_range)
        axs[1].set_ylim(0,n_frame)
        axs[1].set_ylabel("Frame")

        # peakfit
        if flag_fit:
            axs[1].set_xlim(axs[1].get_xlim())
            axs[1].set_ylim(axs[1].get_ylim())
            axs[1].plot(peaks,
                        np.arange(n_frame),
                        marker = "o",
                        c = "tab:orange",
                        ms = 1,
                        mew = 0,
                        lw = 0)
            
            axs[2].plot(peaks,
                        np.arange(n_frame),
                        marker = "o",
                        c = "tab:orange",
                        ms = 3,
                        mew = 0,
                        lw = 0,
                        clip_on = False,
                        )
            avg_peaks = np.average(peaks)
            dmax = np.max(np.abs(avg_peaks - peaks))
            axs[2].set_xlim(avg_peaks - dmax*1.5, avg_peaks + dmax*1.5)
            axs[2].set_ylim(0,n_frame)

            axs[2].set_xlabel("2theta [deg.]")
            axs[2].set_ylabel("Frame")
        else:
            axs[1].set_xlabel("2theta [deg.]")

        # 出力
        imgfilename = os.path.join(cachedir, "disp_caking.png")
        plt.savefig(imgfilename, dpi = 300)
        plt.close()
        print("Save figure: " + imgfilename)

    # figure
    if flag_plotly:
        # plotly

        # figure
        fig = go.Figure()
        trace_indices = dict()

        # Heatmap
        heatmap = go.Heatmap(
            z = np.log(disp_d),
            x = disp_tth,
            y = np.arange(n_frame),
            colorscale = "Viridis",
            zauto = True,
            visible = True,
            name = "Heatmap",
            colorbar = {
                'title': "Intensity [arb. unit]"
            }
        )
        fig.add_trace(heatmap)
        trace_indices["heatmap"] = len(fig.data) - 1 # type: ignore

        if flag_fit:
            plot_on_heatmap = go.Scatter(
                x = peaks,
                y = np.arange(n_frame),
                mode = "lines+markers",
                marker = {
                    'color': "#ff7f0e",
                    'size': 3,
                    'symbol': "circle"
                },
                line = {
                    'color': "#ff7f0e",
                    'width': 1
                },
                name = "Peaks",
                visible = True,
            )
            fig.add_trace(plot_on_heatmap)
            trace_indices["plot_on_heatmap"] = len(fig.data) - 1 # type: ignore

        if flag_fit:
            plot_single = go.Scatter(
                x = np.arange(n_frame),
                y = peaks,
                mode = "lines+markers",
                marker = {
                    'color': "#1f77b4",
                    'size': 3,
                    'symbol': "circle"
                },
                line = {
                    'color': "#1f77b4",
                    'width': 1
                },
                name = "Peaks",
                visible = False,
            )
            fig.add_trace(plot_single)
            trace_indices["plot_single"] = len(fig.data) - 1 # type: ignore

        # animation
        raw_anim = go.Scatter(
            x = tth[fit_mask],
            y = d[0][fit_mask],
            mode = "markers",
            marker = dict(
                color = "#1f77b4",
                size = 3,
                symbol = "circle"
            ),
            name = "Raw",
            visible=False
        )
        fig.add_trace(raw_anim)
        trace_indices["raw_anim"] = len(fig.data) - 1 # type: ignore

        if flag_fit:
            fit_anim = go.Scatter(
                x = xs,
                y = pseudoVoigt(xs, *(popts[0])),
                mode = "lines",
                line = dict(
                    color = "#ff7f0e",
                    width = 1
                ),
                name = "Fit",
                visible = False
            )
            fig.add_trace(fit_anim)
            trace_indices["fit_anim"] = len(fig.data) - 1 # type: ignore
        
        # frameを設定
        if flag_fit:
            frames_list = [
                go.Frame(
                    data = [
                        go.Scatter(
                            y = d[k][fit_mask]
                        ),
                        go.Scatter(
                            y = pseudoVoigt(xs, *(popts[k]))
                        )
                    ],
                    name = str(k),
                    traces = [
                        trace_indices["raw_anim"],
                        trace_indices["fit_anim"]
                    ],
                ) for k in range(n_frame)
            ]
        else:
            frames_list = [
                go.Frame(
                    data = [
                        go.Scatter(
                            y = d[k][fit_mask]
                        )
                    ],
                    name = str(k),
                    traces = [
                        trace_indices["raw_anim"]
                    ]
                ) for k in range(n_frame)
            ]
        fig.frames = frames_list

        # 可視化リストを生成する関数を作成
        def get_visibility(active_names):
            vis = [False] * len(fig.data) # type: ignore
            for name in active_names:
                if name in trace_indices.keys():
                    vis[trace_indices[name]] = True
            return vis
        
        button_change_figure = [
            dict(
                label = "Heatmap",
                method = "update",
                args = [
                    {'visible': get_visibility([
                        "heatmap",
                        "plot_on_heatmap",
                    ])},
                    {'title': "Heatmap",
                    'xaxis': {
                        'title': "2theta [deg.]",
                        "range": rad_range,
                        'autorange': False
                    },
                    'yaxis': {
                        'title': "Frame",
                        'range': (0, n_frame),
                        'autorange': False,
                    },
                    'showlegend': False,
                    'updatemenus[1].visible': False,
                    'sliders[0].visible': False
                    }
                ]
            ),
            dict(
                label = "Animation",
                method = "update",
                args = [
                    {'visible': get_visibility([
                        "raw_anim",
                        "fit_anim",
                    ])},
                    {'title': "Animation",
                    'xaxis': {
                        'title': "2theta [deg.]",
                        "range": fit_range,
                        'showline': True,
                        'mirror': "all",
                        'linewidth': 1,
                        'linecolor': "#777777",
                        'ticks': "inside",
                        'ticklen': 5,
                        'type': "linear",
                    },
                    'yaxis': {
                        'title': "Intensity [arb. unit]",
                        'showline': True,
                        'mirror': "all",
                        'linewidth': 1,
                        'linecolor': "#777777",
                        'ticks': "inside",
                        'ticklen': 5,
                        'type': "linear",
                    },
                    'showlegend': True,
                    'updatemenus[1].visible': True,
                    'sliders[0].visible': True,
                    }
                ]
            ),
        ]
        if flag_fit:
            button_change_figure.append(
                dict(
                    label = "Peak position",
                    method = "update",
                    args = [
                        {'visible': get_visibility(["plot_single"])},
                        {'title': "Peaks",
                        'xaxis': {
                            'title': "Frame",
                            'showline': True,
                            'mirror': "all",
                            'linewidth': 1,
                            'linecolor': "#777777",
                            'ticks': "inside",
                            'ticklen': 5,
                            'type': "linear",
                        },
                        'yaxis': {
                            'title': "2theta [deg.]",
                            'showline': True,
                            'mirror': "all",
                            'linewidth': 1,
                            'linecolor': "#777777",
                            'ticks': "inside",
                            'ticklen': 5,
                            'type': "linear",
                        },
                        'showlegend': False,
                        'updatemenus[1].visible': False,
                        'sliders[0].visible': False
                        }
                    ]
                ),
            )
        updatemenus_change_figure= dict(
            dict(
                buttons = button_change_figure,
                direction = "down",
                x = 0,
                y = 1.02,
                xanchor = "left",
                yanchor = "bottom"
            )
        )
        updatemenus_frame = dict(
            type='buttons',
            direction = 'left', # 横並びの場合 'left', 縦並びの場合 'down'
            showactive=False,
            active = -1,
            y=0,
            x=0,
            xanchor='left',
            yanchor='top',
            pad=dict(t=65, r=0),
            buttons=[
                dict(
                    label='▶︎',
                    method='animate',
                    args=[
                        None,
                        dict(
                            frame=dict(
                                duration=100,
                                redraw=True
                            ),
                            fromcurrent=True,
                            mode='immediate'
                        )
                    ]
                ),
                dict(
                    label='⏸',
                    method='animate',
                    args=[
                        [None],
                        dict(
                            frame=dict(
                                duration=0,
                                redraw=False
                            ),
                            mode='immediate',
                            transition=dict(duration=0)
                        )
                    ]
                )
            ],
            visible = False
        )

        # slider
        slider = dict(
            steps=[dict(
                method='animate',
                args=[
                    [str(k)],
                    dict(
                        mode='immediate',
                        frame=dict(
                            duration=0,
                            redraw=True
                            ),
                        transition=dict(
                            duration=0
                        )
                    )
                ],
                label=str(k),
            ) for k in range(n_frame)],
            ticklen = 0,
            minorticklen = 0,
            active=0,
            y=0,
            x=0, # スライダーの開始位置
            len=1, # スライダーの長さ
            xanchor='left',
            yanchor='top',
            pad=dict(
                b=10,
                t=50,
                l=100
            ),
            currentvalue=dict(
                prefix='Frame = ',
                visible=True,
                xanchor='left'
            ),
            visible = False,
        )

        fig.update_layout(
            title = dict(
                text = "Heatmap",
                font = dict(
                    size = 26,
                    color = "#777777"
                ),
                x = 0.05,
                y = 0.95,
                xanchor = "left",
                yanchor = "top",
            ),
            xaxis = {'title': "2theta [deg.]"},
            yaxis = {'title': "Frame"},
            plot_bgcolor = '#ffffff',
            updatemenus = [
                updatemenus_change_figure,
                updatemenus_frame,
            ],
            sliders = [slider],
            showlegend = False,
            legend = dict(
                bordercolor = "#000000",
                orientation = "h",
                yanchor = "bottom",
                y = 1,
                bgcolor = "rgba(255,255,255,0)",
                xanchor = "center",
                x = 0.5
            ),
            width = 800,
            height = 600,
        )

        fig.show()
    else:
        display(Image(imgfilename, width = 600))
        
    return
disp_caking(hdf_disp_caking, rad_range, flag_fit, fit_range, flag_plotly)
del disp_caking

Inputs:
	hdf:         D:\DATA\SPring-8-2025-Dec\Analysis\FPD\create_hdfdata\.cache\UODE36_0024_integrate1d.hdf
	rad_range:   11-14
	flag_fit:    False
	fit_range:   11.9-12.3
	flag_plotly: True
Save figure: d:\DATA\SPring-8-2025-Dec\Analysis\FPD\create_hdfdata\.cache\disp_caking.png


## csv作成

In [10]:
# hdfファイルを指定
hdf_csv = hdf_1d
hdf_csv = r"D:\DATA\SPring-8-2025-Dec\Analysis\FPD\create_hdfdata\.cache\UODE36_0024_integrate1d.hdf"

# 1枚をcsvにするか、複数のフレームの平均をcsvにするか
flag_1shot = True
# flag_1shot = False

# 1枚の場合、フレームを指定
frame = 39

# 複数フレームの場合、レンジを指定
frame_range = (51,56)


In [11]:
# PDIndexer用にcsvを保存する
def save_csv(hdf: str,
             flag_1shot: bool,
             frame: int,
             frame_range: tuple[int, int]
             ) -> str:

    ## 1枚の場合
    if flag_1shot:
        with h5py.File(hdf, mode = "r") as f:
            tth = np.array(f["rad"][()]) # type: ignore
            insty = np.array(f["integrated/frame = {}".format(frame)][()]) # type: ignore
        csvfilename = os.path.join(cachedir, i2a.name + "_{}.csv".format(frame))

    ## 複数フレーム
    else:
        with h5py.File(hdf, mode = "r") as f:
            tth = np.array(f["rad"][()]) # type: ignore
            insty = np.zeros(shape = tth.shape)
            for i in range(*frame_range):
                insty += np.array(f["integrated/frame = {}".format(i)][()]) # type: ignore
            insty /= (frame_range[1] - frame_range[0])
        csvfilename = os.path.join(cachedir, i2a.name + "_{}-{}.csv".format(*frame_range))

    ## 保存
    df = pd.DataFrame([tth, insty]).T
    df.to_csv(csvfilename, index = False, header = False)
    print(csvfilename)
    display(df)

    return csvfilename
csvfilename = save_csv(hdf_csv, flag_1shot, frame, frame_range)
del save_csv

d:\DATA\SPring-8-2025-Dec\Analysis\FPD\create_hdfdata\.cache\UODE36_0023_39.csv


,0,1
0,3.002700,30.710297
1,3.008100,28.821835
2,3.013500,29.098782
3,3.018900,29.695175
4,3.024300,30.010288
5,3.029700,28.483332
6,3.035100,28.493999
7,3.040500,30.058493
8,3.045900,27.937553
9,3.051300,29.593899


## Azumuthal方向切り開き

In [10]:
# 切り開きステップ
npt_azim = 512
npt_azim = 2048

# 切り開き範囲
radial_range = (13.55, 14)

# 表示するのはどちらか（matplotlibはいずれにせよ保存されます）
kind = "plotly"
# kind = "matplotlib"

# peakをトラッキングする
peak_tracking = True
# peak_tracking = False

# peak情報が書かれたcsv
peak_csv = r"D:\DATA\SPring-8-2025-Dec\Analysis\run2\UODE36_0024\UODE36_0024_hcp011_popts.csv"

# peakトラッキングをする場合のradial方向の積分幅 [deg.]
peak_width = 0.05

In [16]:
# Azumuthal方向の切り開きを実施し、hdfファイルを保存する
def integrate_radial(npt_azim: int,
                     radial_range: tuple,
                     poni: str,
                     kind: str,
                     peak_tracking: bool,
                     peak_csv: str,
                     peak_width: float) -> dict:
    
    if peak_tracking:
        PeakPosi = pd.read_csv(
            peak_csv,
            usecols = [4],
        ).values
        arr_width = np.array([-0.5, 0.5]) * peak_width

    # インスタント化
    ai = pyFAI.load(poni)

    # データ格納用辞書
    d = dict()
    
    for i,f in enumerate(tqdm(i2a.filelist)):

        if peak_tracking:
            rrange = arr_width + PeakPosi[i]
        else:
            rrange = radial_range

        # データ読み込み
        hisdata = his2array(f)

        # 切り開き
        chi, insty = ai.integrate_radial(
            hisdata,
            npt = npt_azim,
            radial_unit = "2th_deg",
            method = "ocl",
            radial_range = rrange
        )

        # データ格納
        if not i:
            d["chi"] = chi
            d["data"] = []    
        d["data"].append(insty)

    # データ格納
    d["data"] = np.vstack(d["data"])
    hdffilename = os.path.join(cachedir, i2a.name + "_azim.hdf")
    with h5py.File(hdffilename, mode = "w") as f:
        f.create_dataset(
            name = "chi",
            data = d["chi"],
            dtype = d["chi"].dtype,
            shape = d["chi"].shape
        )
        f.create_dataset(
            name = "value",
            data = d["data"],
            dtype = d["data"].dtype,
            shape = d["data"].shape
        )
    print("[Save hdf]: " + hdffilename)

    if True:
        # figure作成
        fig = plt.figure()
        fig.set_size_inches(8,4.5)

        # imshowの作成
        ax = fig.add_axes(
            rect = (0.1,0.1,0.68,0.8)
        )
        cmap = ax.imshow(
            d["data"].T,
            aspect = "auto",
            extent = (
                -0.5,
                d["data"].shape[0] + 0.5,
                d["chi"][0] + (d["chi"][1]-d["chi"][0])/2,
                d["chi"][-1] + (d["chi"][1]-d["chi"][0])/2
            ),
            origin = "lower",
            norm = matplotlib.colors.LogNorm(), # type: ignore
        )
        if peak_tracking:
            title = i2a.name + " (peak tracking)"
        else:
            title = i2a.name + " (2theta: {}-{})".format(*radial_range)
        ax.set_title(title,
                    fontsize = 12,
                    loc = "left")
        ax.tick_params(direction = "out")
        ax.set_xlabel("Frame", fontsize = 14)
        ax.set_ylabel("Azimuithal angle [deg.]", fontsize = 14)

        # colorbarの作成
        cbar = fig.add_axes(
            rect = (0.8,0.1,0.02,0.8)
        )
        matplotlib.colorbar.Colorbar( # type: ignore
            mappable = cmap,
            ax = cbar,
            orientation = "vertical"
        )
        cbar.set_ylabel("Intensity [arb. unit]", fontsize = 10)
        cbar.tick_params(direction = "out")

        # 出力
        pngfilename = os.path.join(cachedir, i2a.name + "_azim.png")
        plt.savefig(pngfilename, dpi = 300)
        plt.close()
        print("[Save figure]: " + pngfilename)

    if kind == "plotly":

        # figure
        fig = go.Figure()

        # Heatmap
        heatmap = go.Heatmap(
            z = np.log(d["data"].T),
            y = d["chi"],
            colorscale = "Viridis",
            zauto = True,
            visible = True,
            name = "Heatmap",
            colorbar = {
                'title': "Intensity [arb. unit]"
            }
        )
        fig.add_trace(heatmap)


        fig.update_layout(
            title = dict(
                text = title,
                font = dict(
                    size = 18,
                    color = "#222222"
                ),
                x = 0.05,
                y = 0.9,
                xanchor = "left",
                yanchor = "top",
            ),
            xaxis = {'title': {"text": "Frame",
                            "font": {"size": 20,
                                        "color": "#222222"}}},
            yaxis = {'title': {"text": "Azumuthal angle [deg.]",
                            "font": {"size": 20,
                                        "color": "#222222"}}},
            plot_bgcolor = '#ffffff',
            showlegend = False,
            legend = dict(
                bordercolor = "#000000",
                orientation = "h",
                yanchor = "bottom",
                y = 1,
                bgcolor = "rgba(255,255,255,0)",
                xanchor = "center",
                x = 0.5
            ),
            width = 800,
            height = 600,
        )
        fig.show()

    elif kind == "matplotlib":
        display(Image(filename = pngfilename, width = 800))

    with h5py.File(hdffilename, mode = "r") as f:
        h5_tree(f)

    return d

# 例外処理
if not peak_tracking:
    peak_csv = ""
    peak_width = 0

# 演算
d_integrate_radial = integrate_radial(npt_azim, radial_range, poni, kind, peak_tracking, peak_csv, peak_width)
del integrate_radial

100%|██████████| 133/133 [00:07<00:00, 18.56it/s]


[Save hdf]: d:\DATA\SPring-8-2025-Dec\Analysis\FPD\create_hdfdata\.cache\UODE36_0024_azim.hdf
[Save figure]: d:\DATA\SPring-8-2025-Dec\Analysis\FPD\create_hdfdata\.cache\UODE36_0024_azim.png


<HDF5 file "UODE36_0024_azim.hdf" (mode r)>
├── chi ((2048,), float64)
└── value ((133, 2048), float32)


## Unroll

In [11]:
# 散乱角方向のステップ
npt_step = 0.005

# 方位角方向
npt_azim = 1024
npt_azim = 2048

# 切り開き範囲
radial_range = (12,14)
radial_range = (13.55, 14)
# radial_range = (12.5,13.5)

In [ ]:
# unrollを行う
hdf_2d = i2a.integrate2D(
    poni = poni,
    npt_rad = (radial_range[-1]-radial_range[0]) / npt_step,
    npt_azim = npt_azim,
    radial_range = radial_range,
)

In [ ]:
# unrollの結果を表示する（plotly）
## http://127.0.0.1:8025/ にアクセスしてください
## python "C:\Users\okaza\Documents\Documents\fpd\dash_Integrated2D\Dash_Integrated2D.py"

## Tiff画像の出力

In [78]:
# 保存するフレーム
frame_range = (96,108)
frame_range = (0,16)

In [79]:
# Tiff画像の出力
def save_tiff(frame_range: tuple[int, int]) -> str:

    # 引数の出力
    print("=" * 30)
    print("Input:")
    print("\tframe_range: {}-{}".format(*frame_range))
    print("=" * 30)

    # imgファイルの生成
    hisdata = []
    for j in range(*frame_range):
        f = i2a.filelist[j]
        hisdata.append(his2array(f))
    img = np.average(np.stack(hisdata), axis = 0)

    ## 保存
    imgfilename = os.path.join(cachedir, header + "{}-{}.tif".format(*frame_range))
    im.fromarray(img).save(imgfilename)
    print("[Save tiff]: " + imgfilename)

    return imgfilename
save_tiff(frame_range)
del save_tiff

Input:
	frame_range: 0-16
[Save tiff]: d:\DATA\SPring-8-2025-Dec\Analysis\FPD\create_hdfdata\.cache\UODE36_16_0-16.tif
